In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

一个完整的Agent至少要包含两个关键的部分：
- **模型**：是Agent的大脑，负责推理、分析，规划任务步骤
- **工具**：是Agent的手脚，负责执行任务，与外界交互

因此，定义带有工具的Agent的基本流程如下：
- 定义工具
- 初始化模型
- 初始化Agent，绑定模型和工具

# 1.自定义工具

所谓的**工具（Tool）**，本质就是一个可调用的**函数**，但是这个函数不是我们自己去调用，而是给模型调用。因此除了定义函数外，我们还需要清晰描述这个工具，让模型知道这个工具如何使用。包括下列信息：
- 工具名
- 工具的作用
- 工具需要的参数


## 1.1.基于tool描述工具
在LangChain中，定义工具需要用到@tool装饰器，我们可以通过装饰器来定义工具名、工具的作用：


In [2]:
from langchain_core.tools import tool

@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

## 1.2.使用函数名和文档注释描述工具

如果不@tool装饰器没有定义工具名和作用描述，此时：
- 工具名：默认就是函数名
- 工具所需的参数：默认就是函数的参数列表
- 工具作用的描述：默认就是函数的文档注释

In [3]:
from langchain_core.tools import tool
# 通过tool装饰器定义工具
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [4]:
# 定义一个查询天气的tool
@tool
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    Get current weather and optional forecast.
    Args:
        location: city name or coordinates
        units: unit of degrees
        include_forecast: does it include the weather forecast
    """
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

## 1.3.定义Pydantic Model描述参数
如果函数的参数比较多，而且比较复杂，此时建议通过pydantic model来描述参数列表。


In [5]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal


# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference, default is celsius."
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


工具调用方式与普通函数调用方式一致。


In [7]:
square_root.invoke({"x": 467})

21.61018278497431

In [8]:
get_weather.invoke({"location": "杭州", "include_forecast": True})

'Current weather in 杭州: 22 degrees C\nNext 5 days: Sunny'

## 测试

In [9]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="deepseek-v4-flash",
    tools=[square_root, get_weather],
    system_prompt="你可以使用工具回答用户问题，调用工具时尽量使用默认参数，除非用户特别指定。"
)

In [10]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="杭州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


好的，我来查一下杭州的天气和未来几天的预报。Current weather in 杭州: 22 degrees C
Next 5 days: Sunny杭州的天气情况如下：

### 🌤 当前天气
- **温度**：22°C

### 📅 未来5天预报
- **天气状况**：以**晴天**为主，天气不错！

总体来看，杭州接下来几天天气都很好，阳光充足，适合出行游玩！如果打算出门，建议做好防晒哦～ ☀️

In [11]:
response = agent.invoke(
    {"messages": [HumanMessage(content="467和529的平方根是多少?")]},
)

for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

467和529的平方根是多少?
None
================================== Ai Message ==================================

好的，我来计算467和529的平方根。
Tool Calls:
  square_root (call_00_f1tbC83Tke46uAOX8yQC7194)
 Call ID: call_00_f1tbC83Tke46uAOX8yQC7194
  Args:
    x: 467
  square_root (call_01_0DbzedrTDtxB3nkkJUv05937)
 Call ID: call_01_0DbzedrTDtxB3nkkJUv05937
  Args:
    x: 529
None
================================= Tool Message =================================
Name: square_root

21.61018278497431
None
================================= Tool Message =================================
Name: square_root

23.0
None
================================== Ai Message ==================================

计算结果如下：

- **467的平方根** ≈ **21.6102**
- **529的平方根** = **23**（因为 23² = 529，529是完全平方数）
None


完整流程如图：
<img src="./resources/agent-flow2.png">

# 2.预定义Tool

LangChain中提供了很多预定义的Tool，方便我们使用。例如：
- tavily：就是一个用来做web搜索的工具

## 2.1.基本用法
它的使用步骤是这样的：
- 注册账号，创建API_KEY
- 配置环境变量: TAVILY_API_KEY
- 安装依赖：`uv add langchain-tavily`


In [12]:
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_results=5,
    topic="general", # general, news, finance
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [13]:
search_tool.invoke("蒸蚌是什么梗？")

{'query': '蒸蚌是什么梗？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.douyin.com/shipin/7589558380507842611',
   'title': '蒸蚌小猫梗怎么火的',
   'content': '豪猫，蒸蚌！快奖励我饥荒应有的热度(bushi)原出处@超级无敌大开门 #饥荒 #萝卜纸巾 #蒸蚌. 发布时间：2025-12-24 23:42. ## 相关视频. * 大笨猫又来猜答案了，蒸蚌！来源@超级无敌大开门 #萝卜纸巾 #大笨猫 #哈基米 #青年创作者成长计划. * 【梗百科】萝卜纸巾猫是啥梗？ #超级无敌大开门 #蒸蚌 #鬼灭之刃 #三花猫 #抖音热点记忆2025. 萝卜纸巾猫咪这个呢，来自于一位叫做超级无敌大开门的博主，这个大开门是他家猫的名字。然后呢，这只三花猫呢，是博主从外面捡回来的，看起来呆呆傻傻的。博主呢，也一直在账号里分享他和他家这只绝世好猫平时互动时活蹦乱跳的样子。而这些互动里， 教猫猫抬手坐下、病人物品这类的互动比较居多。而因为博主带着猫咪一起整活，然后这只猫咪呢，嘿嘿也蠢萌蠢萌的，乖的不像话。所 所以博主视频的播放量数据也是越来越好。而最近呢，这类让猫咪辨认物品的视频非常的火，视频中的猫咪看起来并不认识这些物体，以至于只能穷举法，而哪怕穷举对了，博主也会用酷似法务部老鼠的声音疯狂尖叫夸猫咪。萝卜 萝卜真棒！纸巾纸巾真棒！ 由于这段互动又可爱又好玩，于是乎便吸引了诸多网友关注的同时，也用各种各样的方式模仿起了这段萝卜纸巾的互动。萝卜 萝卜真棒！纸巾纸巾真棒！米老鼠真棒！ 米老鼠米老鼠米老鼠真棒！ 高光高光真棒！眼影 眼影真棒！纸巾纸巾 纸巾真棒！萝卜纸巾哪个是萝卜？ what？. 最近纸萝卜纸巾的三花猫视频火了，我们今天就来讲讲宠物的故事。宠物在紫薇斗术里，相当于对应子女宫，子女宫的本质是我所养育、疼爱、付出心血的后代或晚辈，以及与创造新生命情感投射相关的宫位。 在传统社会，这主要是指亲生子女跟在现代社会，很多人不生育或子女成年离家后，宠物刚好扮演了情感子女的角色，主人对他投入的感情、精力、金钱与养育子女非常相似， 所以从相易上，宠物自然可以归入子女宫

In [14]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-v4-flash",
    tools=[search_tool],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

In [15]:
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

蒸蚌是什么梗？
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_00_DJeKmgDvWWIbiyDUMIhw9708)
 Call ID: call_00_DJeKmgDvWWIbiyDUMIhw9708
  Args:
    query: 蒸蚌 梗 含义 出处
================================= Tool Message =================================
Name: tavily_search

{"query": "蒸蚌 梗 含义 出处", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.facebook.com/61586315125667/posts/%E9%9A%A8%E4%BE%BF%E6%8C%87%E4%B8%80%E6%8C%87%E8%81%BD%E5%88%B0%E8%92%B8%E8%9A%8C%E4%BB%A3%E8%A1%A8%E6%8C%87%E5%B0%8D%E4%BA%86%E8%B2%93%E7%8B%97%E5%90%90%E6%A7%BD%E4%B8%BB%E4%BA%BA-%E8%B2%93%E7%8B%97%E9%9B%BB%E5%8F%B0-%E7%B1%B3%E8%80%81%E9%BC%A0%E7%9A%84%E6%84%8F%E6%80%9D-%E8%92%B8%E8%9A%8C-%E6%90%9E%E7%AC%91/122096350329210504", "title": "聽到蒸蚌代表指對了😏 #貓狗吐槽主人#貓狗電台#米老鼠的 ...", "content": "這是在講一個最近很紅的網路貓哏，拿兩個東 西放在貓前面，問牠OOO在哪裡，基本上貓都 會亂指，如果剛好矇

## 2.2.优化

目前的搜索智能体存在两个问题：
- 官方默认的tavily工具过于复杂
- 结果中不包含网页数据源，可信度低

解决思路：
- 自定义tavily工具
- 结构化输出

### 自定义tavily工具

LangChain官方提供的tavily工具包含了完整的参数列表，会导致额外的流量和Token消耗。因此，对于简单的业务，我们建议大家利用tavily自定义工具。


In [16]:
# 先使用官方的客户端做初始化
tavily = TavilySearch(
    max_results=5,
    topic="general"
)

# 然后自己封装为tool
@tool
def web_search(query: str):
    """Search the web for information"""
    return tavily.invoke(query)

### 定义结构化输出实体


In [17]:
from pydantic import BaseModel, Field

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

In [18]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-v4-flash",
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo
)

In [19]:
# 调用agent
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
)

# 获取结构化输出
print(response['structured_response'])

BadRequestError: Error code: 400 - {'error': {'message': 'This response_format type is unavailable now', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}